举例1：不用RAG给LLM灌输上下文数据

In [1]:
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")
# 调用
response = llm.invoke("北京有什么著名的建筑？")
print(response.content)

北京作为中国的首都，拥有许多著名的建筑，以下是一些最具代表性的建筑：

1. **故宫**：也称紫禁城，是明清两代的皇宫，现为故宫博物院，拥有丰富的历史和文化。

2. **天安门广场**：世界上最大的城市广场，广场中央是伟大领袖毛主席的纪念堂，南面是天安门城楼。

3. **天坛**：以其独特的建筑风格和历史背景闻名，是明清两代皇帝祭天祈谷的地方。

4. **颐和园**：皇家园林，以其湖泊和山丘的自然风光，以及古典建筑着称。

5. **圆明园**：虽然大部分建筑在历史上遭到摧毁，但圆明园的遗址仍然显示出其昔日的辉煌。

6. **国家大剧院**：外形独特，像一颗巨蛋的建筑，常用于举办各类文化活动和演出。

7. **鸟巢（国家体育场）**：为2008年北京奥运会而建，具有独特的外观设计，是现代建筑的代表。

8. **水立方（国家游泳中心）**：同样是2008年奥运会的场馆，以其独特的外形和蓝色灯光闻名。

9. **798艺术区**：由老工业区改建而成，聚集了众多当代艺术展览和文化创意产业。

10. **中国国家博物馆**：位于天安门广场东侧，是展示中国历史与文化的重要场所。

这些建筑不仅体现了北京的历史和文化，也展示了现代建筑的成就。


情况2：使用RAG给LLM灌输上下文数据

In [3]:
# 1. 导入所有需要的包
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
import os
import dotenv
dotenv.load_dotenv()
# 2. 创建自定义提示词模板
prompt_template = """请使用以下提供的文本内容来回答问题。仅使用提供的文本信息，如果文本中
没有相关信息，请回答"抱歉，提供的文本中没有这个信息"。
文本内容：
{context}
问题：{question}
回答：
"
"""
prompt = PromptTemplate.from_template(prompt_template)

# 3. 初始化模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

# 4. 加载文档
loader = TextLoader("./asset/load/10-test_doc.txt", encoding='utf-8')
documents = loader.load()

# 5. 分割文档
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
texts = text_splitter.split_documents(documents)
#print(f"文档个数:{len(texts)}")

# 6. 创建向量存储
vectorstore = FAISS.from_documents(
    documents=texts,
    embedding=embedding_model
)

# 7.获取检索器
retriever = vectorstore.as_retriever()
docs = retriever.invoke("北京有什么著名的建筑？")

# 8. 创建Runnable链
chain = prompt | llm

# 9. 提问
result = chain.invoke(input={"question":"北京有什么著名的建筑？","context":docs})
print("\n回答:", result.content)


回答: 北京有以下著名的建筑：

1. 故宫
2. 天安门
3. 颐和园
4. 天坛
5. 长城（八达岭段）
6. 国家体育场（鸟巢）
7. 中央电视台总部大楼
8. 国家大剧院
9. 北京大兴国际机场
10. 鼓楼和钟楼
